# Аугментации обучения

Два потока из `finetune/data_utils.py`. Слева исходный кроп, дальше несколько случайных срабатываний одного и того же преобразования. Полный ряд в конце — то, что реально получает студент: два вида одной пары.

Ч/б BT.601 не входит в эти преобразования. Его применяет энкодер уже после аугментаций.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline
import torch
from PIL import Image
from torchvision import transforms

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "pipeline.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Запускайте ноутбук из репозитория CeramicNet")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from finetune.data_utils import (
    GrayscaleBT601,
    book_transform,
    load_book_pil,
    load_books,
    load_real_crops,
    load_real_pil,
    real_transform,
)

torch.manual_seed(7)
plt.rcParams["axes.unicode_minus"] = False


Беру три книжных кропа и три реальных кропа из COCO. Одиночные преобразования достаю из тех же `Compose`, которыми учится модель, чтобы картинки не разъехались с кодом.

In [ ]:
books = [sample for sample in load_books() if sample.path.is_file()]
book_samples = [books[0], books[len(books) // 3], books[2 * len(books) // 3]]
real_samples = load_real_crops()
real_samples = [real_samples[0], real_samples[len(real_samples) // 3], real_samples[2 * len(real_samples) // 3]]

book_images = [(sample.mark_id, load_book_pil(sample.path)) for sample in book_samples]
real_images = [(sample.file_name, load_real_pil(sample)) for sample in real_samples]

book_pipeline = book_transform()
real_pipeline = real_transform()
book_rotation, book_crop, book_blur = book_pipeline.transforms[:3]
real_rotation, real_jitter = real_pipeline.transforms[:2]
real_noise = real_pipeline.transforms[-1]
blur_only = book_blur.transforms[0]

print("Книги:", ", ".join(name for name, _ in book_images))
print("Реал:", ", ".join(name for name, _ in real_images))
print(f"Blur книг применяется с p={book_blur.p}")
print(f"Шум реала std={real_noise.std}")


In [ ]:
def as_image(value) -> Image.Image:
    if isinstance(value, torch.Tensor):
        array = value.detach().clamp(0, 1).mul(255).byte().permute(1, 2, 0).numpy()
        return Image.fromarray(array)
    return value


def show_grid(rows: list[tuple[str, list]], title: str) -> None:
    columns = max(len(images) for _, images in rows)
    fig, axes = plt.subplots(
        len(rows),
        columns,
        figsize=(2.15 * columns, 2.35 * len(rows)),
        squeeze=False,
    )
    for row_index, (label, images) in enumerate(rows):
        for col_index in range(columns):
            axis = axes[row_index][col_index]
            axis.axis("off")
            if col_index >= len(images):
                continue
            axis.imshow(as_image(images[col_index]))
            if col_index == 0:
                axis.set_title(label, fontsize=9, loc="left")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()


## Книги

Поворот `0..180°` с серой заливкой 114. `RandomResizedCrop` меняет масштаб (`0.75..1.0`) и положение окна. `GaussianBlur` лёгкий: ядро 5, sigma `0.1..1.0`, вероятность 0.3.

In [ ]:
show_grid(
    [(name, [image]) for name, image in book_images],
    "Книги, исходный кроп",
)
show_grid(
    [(name, [image] + [book_rotation(image) for _ in range(4)]) for name, image in book_images],
    "Книги: поворот 0–180°, fill 114. Первый кадр — исходник",
)
show_grid(
    [(name, [image] + [book_crop(image) for _ in range(4)]) for name, image in book_images],
    "Книги: кроп scale 0.75–1.0 и сдвиг окна. Первый кадр — исходник",
)
show_grid(
    [(name, [image] + [blur_only(image) for _ in range(4)]) for name, image in book_images],
    "Книги: лёгкий GaussianBlur, ядро 5, sigma 0.1–1.0. Здесь он включён всегда, чтобы его было видно",
)
show_grid(
    [
        (name, [image, book_pipeline(image), book_pipeline(image)])
        for name, image in book_images
    ],
    "Книги: два вида одной пары, полный book_transform. Первый кадр — исходник",
)


## Реал

Поворот `0..360°`: тарелка на фото может лежать как угодно. Дальше только ColorJitter по яркости и контрасту (`0.4`) и лёгкий шум `std 0.02`. Случайного кропа и размытия нет.

In [ ]:
show_grid(
    [(name, [image]) for name, image in real_images],
    "Реал, кроп без аугментации",
)
show_grid(
    [(name, [image] + [real_rotation(image) for _ in range(4)]) for name, image in real_images],
    "Реал: поворот 0–360°, fill 114. Первый кадр — исходник",
)
show_grid(
    [(name, [image] + [real_jitter(image) for _ in range(4)]) for name, image in real_images],
    "Реал: ColorJitter, яркость и контраст 0.4. Первый кадр — исходник",
)
to_tensor = transforms.ToTensor()
show_grid(
    [
        (name, [image] + [real_noise(to_tensor(image)) for _ in range(4)])
        for name, image in real_images
    ],
    "Реал: GaussianNoise std 0.02. Первый кадр — исходник",
)
show_grid(
    [
        (name, [image, real_pipeline(image), real_pipeline(image)])
        for name, image in real_images
    ],
    "Реал: два вида одной пары, полный real_transform. Первый кадр — исходник",
)


## Что видит энкодер

После аугментации `GrayscaleBT601` складывает каналы как `Y = 0.299R + 0.587G + 0.114B` и копирует яркость в три канала. Ниже полный вид из обучения и его ч/б-версия.

In [ ]:
gray = GrayscaleBT601()


def encoder_view(value) -> torch.Tensor:
    tensor = value if isinstance(value, torch.Tensor) else to_tensor(value)
    return gray(tensor.unsqueeze(0))[0]


def color_and_gray(pipeline, image):
    view = pipeline(image)
    return [view, encoder_view(view)]


show_grid(
    [(name, color_and_gray(book_pipeline, image)) for name, image in book_images],
    "Книги: цветной вид аугментации и ч/б, которое идёт в DINOv2",
)
show_grid(
    [(name, color_and_gray(real_pipeline, image)) for name, image in real_images],
    "Реал: цветной вид аугментации и ч/б, которое идёт в DINOv2",
)


## Контраст перед энкодером

После кропа и до ч/б: CLAHE по яркости (`clipLimit 3`, сетка 8×8) и лёгкая резкость. Слева исходник, в центре усиленный кадр, справа то ч/б, которое видит DINOv2.

In [ ]:
import importlib

import cv2
import numpy as np
import src.preprocessing.image as image_prep

importlib.reload(image_prep)
enhance_stamp_contrast = image_prep.enhance_stamp_contrast


def contrast_views(image: Image.Image):
    bgr = cv2.cvtColor(np.asarray(image.convert("RGB")), cv2.COLOR_RGB2BGR)
    enhanced = enhance_stamp_contrast(bgr)
    enhanced_rgb = Image.fromarray(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))
    return [image, enhanced_rgb, encoder_view(enhanced_rgb)]


show_grid(
    [(name, contrast_views(image)) for name, image in book_images],
    "Книги: исходник, CLAHE и резкость, ч/б после контраста",
)
show_grid(
    [(name, contrast_views(image)) for name, image in real_images],
    "Реал: исходник, CLAHE и резкость, ч/б после контраста",
)
